# Lección 18: Asegurando Agentes de IA con Recibos Criptográficos

## Cuaderno Práctico

Este cuaderno guía a través de cuatro ejercicios:

1. **Firma tu primer recibo** para una llamada a una herramienta de agente y verifícalo.
2. **Manipula el recibo** y observa cómo falla la verificación.
3. **Construye una cadena de tres recibos** y confirma la integridad de la cadena.
4. **Envuelve una llamada a herramienta del Microsoft Agent Framework** para que cada acción emita un recibo.

Todas las primitivas criptográficas se importan de bibliotecas bien mantenidas (`pynacl` para Ed25519, `jcs` para JSON canónico RFC 8785, `hashlib` de la biblioteca estándar de Python para SHA-256). La lógica del recibo en sí es Python simple que puedes leer y modificar.

Ejecuta las celdas en orden. Cada sección es corta y autónoma.


## Configuración

Instala las dos dependencias. Ambas tienen licencias permisivas (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Utilidades auxiliares

Estos dos auxiliares manejan la codificación base64url (sin relleno) y el hash SHA-256 de objetos arbitrarios. Mantienen el resto del cuaderno centrado en la lógica del recibo en sí.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.b64decode(s + padding, altchars=b"-_", validate=True)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sección 1: Firma tu primer recibo

Imagina que nuestro agente para **Contoso Travel** acaba de buscar vuelos de Sídney a Los Ángeles para un cliente. Queremos registrar la declaración de la puerta de enlace sobre esta llamada a la herramienta para que un auditor futuro pueda verificarla bajo una clave de emisor fijada por separado.

### Paso 1.1: Generar una clave de firma

En producción, la clave de firma del agente viviría en un módulo de seguridad de hardware (HSM), Azure Key Vault, o una tienda protegida similar. Para esta lección generamos una clave nueva en memoria.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0


### Paso 1.2: Construir la carga útil del recibo

La carga útil contiene todo lo que queremos que el recibo certifique: quién actuó, qué herramienta, con qué argumentos, qué se devolvió, bajo qué política y cuándo. Calculamos el hash de los argumentos y el resultado en lugar de incluirlos en línea para que el recibo no filtre contenido sensible.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Paso 1.3: Firmar y ensamblar el recibo

Tres pasos:

1. Canonicalizar la carga útil usando JCS para que dos implementaciones que produzcan el mismo recibo lógico generen bytes idénticos.
2. Firmar los bytes canónicos directamente con la llave privada Ed25519. PureEdDSA hace el hash del mensaje internamente, por lo que un pre-hash adicional cambiaría el protocolo.

La firma se adjunta luego a la carga útil original para producir el recibo final.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "Eu3MmrOSdGq2DuemkjNSPKfz5xbr18okNeTU11NJu2usnsnKtC-pjq2PZM1Oap-WXdVgYkL4iW6SeWPSZ2M9BQ",
    "public_key": "g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0"
  }
}


### Paso 1.4: Verificar el recibo

La verificación invierte el proceso. Quitamos la firma, recalculamos los bytes canónicos y verificamos la firma solo después de que la clave pública incrustada coincida con una clave fijada por la parte confiable.

Después de que las claves aceptadas se fijan fuera de banda, la verificación no necesita llamada de servicio. El recibo por sí solo prueba la auto-consistencia, no que su clave incrustada pertenezca al emisor indicado.


In [7]:
def verify_receipt(receipt: dict, trusted_public_keys: set[str]) -> bool:
    # The signature is a structured object: {"alg", "sig", "public_key"}.
    if not isinstance(receipt, dict):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    try:
        # An inline key proves only self-consistency. Trust is configured out of band.
        public_key = sig_obj.get("public_key")
        signature = sig_obj.get("sig")
        if not isinstance(public_key, str) or not isinstance(signature, str):
            return False
        if public_key not in trusted_public_keys:
            return False
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical_bytes = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(public_key))
        verify_key.verify(canonical_bytes, b64url_decode(signature))
        return True
    except (CryptoError, TypeError, ValueError, KeyError, OverflowError):
        return False

is_valid = verify_receipt(receipt, TRUSTED_PUBLIC_KEYS)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, TRUSTED_PUBLIC_KEYS)}")

Receipt is valid: True
Pre-hashed receipt valid: False


Debes ver `Receipt is valid: True` y `Pre-hashed receipt valid: False`. El caso positivo demuestra que la ruta de firma directa-JCS funciona; el control negativo hace que la regla del alcance de la firma sea ejecutable en lugar de dejarla como prosa.


## Sección 2: Manipular el recibo

La razón principal de los recibos es que son evidentes ante manipulaciones. Demostremos esto.

Modificaremos exactamente un carácter del recibo y observaremos que la verificación falla.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, TRUSTED_PUBLIC_KEYS)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### ¿Qué acaba de pasar?

Cuando cambiamos `policy_id`, los bytes canónicos cambiaron. La firma (que estaba sobre los bytes canónicos originales) ya no coincide. La verificación correctamente devuelve `False`.

No hay forma de modificar un campo firmado y mantener la firma original válida sin la clave privada. Un verificador también debe fijar la clave pública aceptada fuera del recibo; publicar o insertar una clave por sí solo no establece confianza.

Pruébalo tú mismo: modifica el `tool_name` o `agent_id` o `timestamp` en la celda de arriba y ejecútalo de nuevo. Cada cambio produce un recibo inválido.


## Sección 3: Encadenar recibos juntos

Un solo recibo protege una acción. La mayoría de los agentes realizan muchas acciones. Para hacer que toda la secuencia sea evidente de manipulación, enlazamos cada recibo con el anterior incluyendo el hash del recibo previo en la carga útil del nuevo recibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Si alguien elimina o reordena un recibo dentro de la secuencia presentada, el siguiente enlace se rompe. Detectar truncamiento o reescritura del sufijo controlada por firmante también requiere una cabeza actual anclada, un punto de control externo, o un compromiso autenticado de población.


In [9]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [10]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:d54d1e138bb080dd6bd825d9f015efc74ad6c7f78ee9504d0b68b39d0733b39c
Receipt 2: tool=confirm_booking, prev=sha256:9fbe5dc2ffcffd7d167ef13a744aecd578e474fb536cb7104ddb1a359607f2f6


In [11]:
def verify_chain(chain: list, trusted_public_keys: set[str]) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt, trusted_public_keys)

        fields = receipt if isinstance(receipt, dict) else {}
        chain_ok = False
        try:
            if i == 0:
                chain_ok = "previous_receipt_hash" in fields and fields["previous_receipt_hash"] is None
            elif isinstance(chain[i - 1], dict):
                expected = receipt_hash(chain[i - 1])
                chain_ok = fields.get("previous_receipt_hash") == expected
        except (TypeError, ValueError, OverflowError):
            pass  # An unhashable/malformed predecessor cannot establish a chain link.

        seq_ok = type(fields.get("sequence")) is int and fields["sequence"] == i
        shape_ok = isinstance(fields.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": fields.get("tool_name", "<malformed>"),
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, TRUSTED_PUBLIC_KEYS):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Ahora rompe la cadena manipulando el recibo del medio y vuelve a verificar. El recibo manipulado no pasa la verificación de su firma, Y el recibo siguiente no pasa la verificación del enlace de la cadena (porque su `previous_receipt_hash` ya no coincide con el hash del recibo del medio modificado).


In [12]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, TRUSTED_PUBLIC_KEYS):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


El recibo 0 aún verifica (no fue modificado y no tiene un predecesor del que depender). El recibo 1 falla en su verificación de firma porque cambiamos `tool_args_hash`. El recibo 2 falla en su verificación de enlace en cadena porque su `previous_receipt_hash` se calculó contra el recibo 1 original (ahora modificado). 

Incluso si un atacante vuelve a firmar el recibo 1 modificado (lo que no puede hacer sin la clave privada), la discrepancia del enlace en cadena en el recibo 2 aún expondría la manipulación. Para ocultar el cambio, el atacante tendría que volver a firmar cada recibo desde el punto de modificación en adelante, lo que requiere la posesión de la clave privada.


## Sección 4: Envolver una llamada a una herramienta de agente con firma de recibo

En un despliegue real, no quieres que cada autor de agente recuerde llamar a `make_receipt`. Quieres que la firma de recibos sea automática para cada invocación de herramienta.

Aquí está el patrón más simple: una clase envoltorio que toma cualquier función de herramienta callable y devuelve una versión que emite recibos. Esto se adapta a cualquier marco de agente, incluyendo el Microsoft Agent Framework (`agent_framework.foundry`).

Si no tienes un proyecto Microsoft Foundry configurado, el mock local a continuación aún demuestra el patrón.


In [13]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [14]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, TRUSTED_PUBLIC_KEYS):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integración con el Microsoft Agent Framework

El envoltorio `ReceiptedTool` anterior es independiente del framework. Para usarlo dentro de un agente construido con Microsoft Agent Framework, registra la función envuelta como una herramienta. Un esquema (reemplazarías el mock con un registro real de herramienta de Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

El framework del agente no necesita saber nada sobre recibos. La firma del recibo está envuelta alrededor de la herramienta, no integrada en el framework. Así es como agregas procedencia al código de agente existente sin reescribir el agente.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline under a relying-party-pinned public key.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
